# CREM: Índice de Gini y Distribución de la Renta P80/P20
**Propósito**: Extrae la información sobre el índice de Gini y la distribución de la renta P80/P20 por municipios a partir de la página oficial del CREM (Centro Regional de Estadística de Murcia).

**Particularidades de esta tabla**:
- La cabecera tiene **dos filas**: la primera agrupa sub-tablas por nombre (con `colspan`), la segunda contiene los años.
- El nombre de columna resultante es `{año}_{nombre_subtabla}` normalizado.
- Se filtran únicamente las filas de nivel **Municipio** (`level2`).

In [1]:
import sys
import re
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
url_base   = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec32.html"
table_name = "raw.crem.indice_gini_y_distribucion_renta_p80_p20"

## Descarga y Parseo del Contenido HTML directamente en Memoria

In [3]:
html = crem.get_html(url_base)
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")
if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# ---------------------------------------------------------------------------
# Parseo de cabecera de dos filas:
#   Fila 0: grupos con colspan  → nombre del grupo (sub-tabla)
#   Fila 1: años                → valor de cada columna de datos
# Resultado: lista de etiquetas compuestas  "{año}_{Grupo}"
# ---------------------------------------------------------------------------
thead     = table.find("thead", id="cabeceraTablaDatos") or table.find("thead")
hdr_rows  = thead.find_all("tr")

# Fila 0: grupos con colspan
grupos = []
for th in hdr_rows[0].find_all("th"):
    texto = th.get_text(strip=True)
    if not texto:
        continue
    span = int(th.get("colspan", 1))
    grupos.extend([texto] * span)

# Fila 1: años
anios = [th.get_text(strip=True) for th in hdr_rows[1].find_all("th") if th.get_text(strip=True)]

columnas = [f"{a}_{g}" for a, g in zip(anios, grupos)]
print(f"Columnas detectadas ({len(columnas)}): {columnas[:6]} …")

Columnas detectadas (18): ['2023_Índice de Gini', '2022_Índice de Gini', '2021_Índice de Gini', '2020_Índice de Gini', '2019_Índice de Gini', '2018_Índice de Gini'] …


In [4]:
# ---------------------------------------------------------------------------
# Parseo de filas: solo level2 (Municipio)
# ---------------------------------------------------------------------------
tbody = table.find("tbody")

mapping_niveles = {1: "Región", 2: "Municipio", 3: "Distrito", 4: "Sección"}

datos_filas = []
for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue

    # Detectar nivel
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break

    # Solo Municipios
    if mapping_niveles.get(level_num) != "Municipio":
        continue

    label_clean = crem.clean_and_decode(th.get_text(strip=True))
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    nombre = match.group(1).strip() if match else label_clean
    if ',' in nombre:
        nombre = nombre.split(',')[1].strip() + ' ' + nombre.split(',')[0].strip()

    tds    = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            values.append(float(val_str) if val_str not in ("", "-", "..") else None)
        except ValueError:
            values.append(None)

    if len(values) == len(columnas):
        registro = {"nombre": crem.normalize_name(nombre)}
        for col, val in zip(columnas, values):
            registro[col] = val
        datos_filas.append(registro)

print(f"Municipios parseados: {len(datos_filas)}")

Municipios parseados: 45


## Integración con PySpark y Delta Lake

In [5]:
spark = tfm_utils.get_spark_session(app_name="CREM_Gini_P80P20")

df = spark.createDataFrame(datos_filas)

# Normalizar columnas (minúsculas, sin acentos)
df_normalized = tfm_utils.normalize_df(df)
tfm_utils.display(df_normalized)

,2015_distribucion_de_la_renta_p80_p20,2015_indice_de_gini,2016_distribucion_de_la_renta_p80_p20,2016_indice_de_gini,2017_distribucion_de_la_renta_p80_p20,2017_indice_de_gini,2018_distribucion_de_la_renta_p80_p20,2018_indice_de_gini,2019_distribucion_de_la_renta_p80_p20,2019_indice_de_gini,2020_distribucion_de_la_renta_p80_p20,2020_indice_de_gini,2021_distribucion_de_la_renta_p80_p20,2021_indice_de_gini,2022_distribucion_de_la_renta_p80_p20,2022_indice_de_gini,2023_distribucion_de_la_renta_p80_p20,2023_indice_de_gini,nombre
0,2.6,31.8,2.6,30.9,2.3,29.3,2.4,28.3,2.3,28.0,2.4,28.4,2.4,28.1,2.3,27.9,2.3,27.4,abanilla
1,2.2,28.8,2.3,28.4,2.2,27.1,2.3,27.8,2.2,26.6,2.3,26.9,2.2,26.9,2.3,27.4,2.2,26.8,abaran
2,2.9,32.9,2.6,32.1,2.4,31.0,2.5,30.3,2.4,29.6,2.5,30.2,2.6,29.9,2.4,29.5,2.4,29.6,aguilas
3,2.6,32.7,2.5,32.4,2.2,29.2,2.3,27.2,2.2,26.7,2.3,27.7,2.2,26.6,2.3,26.8,2.3,27.2,albudeite
4,3.1,34.3,3.2,34.0,2.9,32.5,2.7,31.8,2.7,31.1,2.7,31.0,2.7,30.6,2.7,30.3,2.6,30.1,alcantarilla
5,2.4,27.2,2.4,28.6,2.3,27.1,2.3,25.9,2.1,24.4,2.1,24.3,2.1,24.5,2.2,24.2,2.2,23.7,aledo
6,2.8,32.6,2.9,32.6,2.7,30.9,2.8,30.4,2.6,29.5,2.6,29.7,2.7,29.8,2.6,29.7,2.5,29.1,alguazas
7,2.7,31.6,2.7,31.7,2.6,29.8,2.7,29.1,2.6,28.8,2.6,28.7,2.5,27.9,2.4,28.2,2.5,27.9,alhama_de_murcia
8,2.9,33.0,2.6,32.6,2.4,30.6,2.5,30.8,2.6,30.5,2.5,30.7,2.7,30.0,2.5,30.1,2.7,30.3,archena
9,2.5,29.6,2.6,29.4,2.2,27.6,2.3,27.8,2.2,27.1,2.3,27.2,2.4,27.1,2.5,27.2,2.3,26.5,beniel


In [6]:
delta_path = tfm_utils.full_table_path(table_name)
print(f"Escribiendo {df_normalized.count()} registros en: {table_name}")
print(f"Destino: {delta_path}")

(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)
print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en: raw.crem.indice_gini_y_distribucion_renta_p80_p20
Destino: /tfm/delta_lake/raw/crem/indice_gini_y_distribucion_renta_p80_p20
¡Escritura en Delta Lake completada con éxito!
